# Manuscript benchmark: 2D triangle L1 (with L2 pre) on every section of the real DVF

This notebook applies the two-stage `L2-multipass -> L1-polish` correction recipe (established in notebooks 17 and 18) to **every z-slice** of the real registration field at `data/corrected_correspondences_count_touching/registered_output/deformation3d.npy`.

**Pipeline per slice.** For each `z in [0, 528)`:

1. Extract the slice as a `(3, 1, H, W)` deformation; channel 0 (dz) is identically zero in this dataset, so each slice is a genuine 2D problem.
2. **L2 phase.** Run `iterative_serial(..., enforce_triangles=True)` -- the package's windowed SLSQP solver, configured to use the 2-triangle constraint (the manuscript's check) rather than central-difference Jdet. Drives `n_neg_tri -> 0`.
3. **L1 polish phase.** Identify connected components of cells *touched* by the L2 phase. For each component, re-solve the bounding sub-window with the smoothed-L1 objective + 2-triangle constraint + frozen-edge boundary. Accept only if no new folds appear and the L1 norm strictly drops.
4. Record one CSV row per slice with init/L2/L1 metrics and timing. Save incrementally so killing the kernel preserves work; on restart the notebook resumes from where it left off.

**Output.** `notebooks/manuscript/output/2d_real_full/`:

- `per_slice.csv` -- the per-slice results table (appended row-by-row).
- `checkpoint.npz` -- snapshot of the corrected `(3, D, H, W)` volume, updated every N slices.
- `summary_*.{pdf,png}` -- aggregate plots once every slice is done.

Companion notebook: `04_benchmark_3d_real_full.ipynb` runs the same recipe with the 6-tet constraint on the full 3D volume.

## Setup

In [ ]:
import os, sys, time, json, traceback
import multiprocessing as mp
sys.path.insert(0, os.path.abspath('../..'))
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import label as cc_label

from dvfopt import DEFAULT_PARAMS
from dvfopt.jacobian import triangle_sign_areas2D
from dvfopt.jacobian.triangle_sign import _triangle_areas_2d

import _bench_worker  # noqa: E402  (sibling module in this folder)

# Tunables.
THRESHOLD = DEFAULT_PARAMS['threshold']  # 0.01
ERR_TOL = DEFAULT_PARAMS['err_tol']      # 1e-5
EPS_L1 = 1e-4

# Windowed outer-loop config (mirrors the 3D notebook). Each outer iter
# picks the worst-fold pixel, builds a connected-component bbox capped at
# MAX_WINDOW_PER_AXIS x MAX_WINDOW_PER_AXIS cells (so SLSQP variable count
# is bounded), and runs a per-window SLSQP in a child process with a
# wall-clock timeout.
MAX_WINDOW_PER_AXIS = 14         # max along any axis (cells, not pixels)
MAX_WINDOW_CELLS = 200           # max total cells in a window
L2_MAX_OUTER = 5000              # outer-loop iter cap per slice
L2_MAX_MINIMIZE_ITER = 250       # per-window SLSQP iter cap
L2_LOCAL_TIMEOUT_S = 30          # per-window SLSQP wall-clock timeout
L2_STALL_LIMIT = 30              # # of consecutive non-improving iters before
                                  # the "perturb-on-stall" fallback fires
L2_PERTURB_SIGMA = 0.02          # sigma for the Gaussian perturb fallback
L2_MAX_PERTURB_ROUNDS = 2        # how many times we may perturb a stalled slice

L1_POLISH_MAX_ITER = 250
L1_LOCAL_TIMEOUT_S = 20
L1_TOUCH_TOL = 1e-3
L1_BORDER = 1
L1_COMPONENT_MAX_CELLS = 200

CHECKPOINT_EVERY = 10
SMOKE_TEST = False
SMOKE_SLICE_INDICES = None

OUTPUT_DIR = os.path.abspath('output/2d_real_full')
os.makedirs(OUTPUT_DIR, exist_ok=True)
CSV_PATH = os.path.join(OUTPUT_DIR, 'per_slice.csv')
CKPT_PATH = os.path.join(OUTPUT_DIR, 'checkpoint.npz')
LOG_PATH = os.path.join(OUTPUT_DIR, 'run.log')

DATA_PATH = os.path.abspath(
    os.path.join('..', '..', 'data',
                 'corrected_correspondences_count_touching',
                 'registered_output', 'deformation3d.npy'))

print(f'THRESHOLD = {THRESHOLD},  EPS_L1 = {EPS_L1}')
print(f'L2 local timeout = {L2_LOCAL_TIMEOUT_S}s, L1 local timeout = {L1_LOCAL_TIMEOUT_S}s')
print(f'window cap: {MAX_WINDOW_PER_AXIS}x{MAX_WINDOW_PER_AXIS} cells (<= {MAX_WINDOW_CELLS} total)')

## Load the real DVF and survey folds per slice

In [ ]:
phi_full = np.load(DATA_PATH)   # (3, D, H, W) with [dz, dy, dx]
D, H, W = phi_full.shape[1:]
print(f'deformation3d.npy shape : {phi_full.shape}  (D={D}, H={H}, W={W})')
print(f'channel ranges:')
for c, name in enumerate(['dz', 'dy', 'dx']):
    print(f'  {name}: min={phi_full[c].min():+.3f}, max={phi_full[c].max():+.3f}, mean={phi_full[c].mean():+.3f}')
if not np.all(phi_full[0] == 0):
    print('NOTE: channel 0 (dz) is NOT identically zero -- per-slice 2D treatment is an approximation.')
else:
    print('Channel 0 (dz) is identically zero -- per-slice 2D treatment is exact.')

In [ ]:
def slice_fold_stats(phi_full, z):
    """Return (n_neg_tri, min_tri, n_neg_jdet) for slice z."""
    dy = phi_full[1, z]; dx = phi_full[2, z]
    T1, T2 = _triangle_areas_2d(dy, dx)
    n_neg = int((T1 <= 0).sum() + (T2 <= 0).sum())
    return n_neg, float(min(T1.min(), T2.min()))


t0 = time.time()
fold_per_slice = np.zeros(D, dtype=np.int64)
min_tri_per_slice = np.zeros(D, dtype=np.float64)
for z in range(D):
    n_neg, mt = slice_fold_stats(phi_full, z)
    fold_per_slice[z] = n_neg
    min_tri_per_slice[z] = mt
print(f'Surveyed {D} slices in {time.time()-t0:.1f}s')
print(f'slices with folds : {int((fold_per_slice > 0).sum())} / {D}')
print(f'total folded triangles across all slices: {int(fold_per_slice.sum())}')
print(f'max folds in a single slice : {int(fold_per_slice.max())}  (slice z={int(np.argmax(fold_per_slice))})')
print(f'global min triangle area    : {float(min_tri_per_slice.min()):+.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 3.4), constrained_layout=True)
axes[0].plot(fold_per_slice, lw=0.8)
axes[0].set_xlabel('z (slice index)'); axes[0].set_ylabel('n_neg_tri (initial)')
axes[0].set_title('Initial folded-triangle count per slice')
axes[1].hist(fold_per_slice[fold_per_slice > 0], bins=60, color='#5b7fb5')
axes[1].set_xlabel('n_neg_tri'); axes[1].set_ylabel('# slices')
axes[1].set_title(f'Distribution over the {int((fold_per_slice>0).sum())} folded slices')
plt.show()

## Helpers

Three building blocks:

1. `metrics_2d` -- common metric block (n_neg_tri, min_tri, L1, L2) given two `(2,H,W)` fields.
2. `run_l2_phase` -- thin wrapper around `iterative_serial(enforce_triangles=True)` that also captures wall time.
3. `run_l1_polish_phase` -- the windowed L1 polish: connected components of touched cells, then per-component SLSQP with the smoothed-L1 objective.

In [ ]:
def metrics_2d(phi_now, phi_anchor):
    """phi_now, phi_anchor: shape (2, H, W). Returns dict."""
    T1, T2 = _triangle_areas_2d(phi_now[0], phi_now[1])
    return {
        'n_neg_tri': int((T1 <= 0).sum() + (T2 <= 0).sum()),
        'min_tri': float(min(T1.min(), T2.min())),
        'L1': float(np.abs(phi_now - phi_anchor).sum()),
        'L2': float(np.linalg.norm(phi_now - phi_anchor)),
    }

In [ ]:
def _run_worker_with_timeout(target, args, timeout_s):
    """Run ``target(*args, send_conn)`` in a child process; kill if it exceeds ``timeout_s``.

    Returns ``(status, payload, info)`` with status in {'ok', 'err', 'timeout'}.
    """
    ctx = mp.get_context('spawn')
    parent_conn, child_conn = ctx.Pipe(duplex=False)
    proc = ctx.Process(target=target, args=(*args, child_conn))
    proc.start()
    child_conn.close()
    t0 = time.time()
    if parent_conn.poll(timeout_s):
        msg = parent_conn.recv()
        proc.join(timeout=5)
        if proc.is_alive():
            proc.terminate(); proc.join(timeout=5)
        return msg[0], msg[1:], {'wall_time': time.time() - t0}
    proc.terminate()
    proc.join(timeout=5)
    if proc.is_alive():
        proc.kill(); proc.join()
    return 'timeout', None, {'wall_time': time.time() - t0}


def _shrink_to_cap_2d(y0, y1, x0, x1, cap_per_axis, cap_total):
    def shrink_axis(a0, a1, cap):
        if a1 - a0 <= cap:
            return a0, a1
        ctr = (a0 + a1) // 2
        half = cap // 2
        return max(a0, ctr - half), min(a1, ctr - half + cap)
    y0, y1 = shrink_axis(y0, y1, cap_per_axis)
    x0, x1 = shrink_axis(x0, x1, cap_per_axis)
    while (y1 - y0) * (x1 - x0) > cap_total:
        if (y1 - y0) >= (x1 - x0) and (y1 - y0) > 2:
            y1 -= 1
        elif (x1 - x0) > 2:
            x1 -= 1
        else:
            break
    return y0, y1, x0, x1


def _build_window_around_2d(focus_cell, labels, cell_fold_mask,
                             *, border=1,
                             cap_per_axis=MAX_WINDOW_PER_AXIS,
                             cap_total=MAX_WINDOW_CELLS):
    cy, cx = focus_cell
    cid = labels[cy, cx]
    if cid == 0:
        y0, y1 = cy, cy + 1
        x0, x1 = cx, cx + 1
    else:
        comp = np.argwhere(labels == cid)
        y0, x0 = comp.min(axis=0)
        y1, x1 = comp.max(axis=0) + 1
    Hc, Wc = labels.shape
    y0 = max(0, y0 - border); y1 = min(Hc, y1 + border)
    x0 = max(0, x0 - border); x1 = min(Wc, x1 + border)
    y0, y1, x0, x1 = _shrink_to_cap_2d(y0, y1, x0, x1, cap_per_axis, cap_total)
    return int(y0), int(y1), int(x0), int(x1)


def _windowed_l2_pass(phi, phi_anchor, *, max_outer, max_minimize_iter,
                       local_timeout_s, stall_limit, log_prefix=''):
    """Run the windowed outer loop until n_neg=0, stall, or budget exhausts.

    Mutates and returns ``phi``. Also returns a stats dict.
    """
    stats = {
        'outer_iters_run': 0,
        'local_timeouts': 0,
        'local_errors': 0,
        'local_successes': 0,
        'stalled': False,
        'window_size_max': 0,
    }
    last_n_neg = None
    stall = 0
    H_, W_ = phi.shape[1:]
    for outer in range(max_outer):
        T1, T2 = _triangle_areas_2d(phi[0], phi[1])
        # Cell-level fold map: a cell is folded if either of its two
        # triangles violates the threshold.
        cell_min = np.minimum(T1, T2)
        cell_fold_mask = cell_min <= THRESHOLD - ERR_TOL
        n_neg = int((T1 <= 0).sum() + (T2 <= 0).sum())
        if n_neg == 0:
            stats['n_neg_after'] = 0
            return phi, stats
        if not cell_fold_mask.any():
            stats['n_neg_after'] = n_neg
            return phi, stats
        labels, _ = cc_label(cell_fold_mask)
        # Worst cell.
        argmin_flat = int(cell_min.argmin())
        wy, wx = np.unravel_index(argmin_flat, cell_min.shape)
        y0, y1, x0, x1 = _build_window_around_2d(
            (wy, wx), labels, cell_fold_mask,
            cap_per_axis=MAX_WINDOW_PER_AXIS, cap_total=MAX_WINDOW_CELLS)
        sy, sx = y1 - y0, x1 - x0
        vsy, vsx = sy + 1, sx + 1
        stats['window_size_max'] = max(stats['window_size_max'], sy * sx)
        # Voxel-corner slices for the (2, H, W) field.
        vy = slice(y0, y1 + 1); vx = slice(x0, x1 + 1)
        phi_win = phi[:, vy, vx].copy()
        phi_anchor_win = phi_anchor[:, vy, vx].copy()
        interior_mask = np.zeros((vsy, vsx), dtype=bool)
        if vsy > 2 and vsx > 2:
            interior_mask[1:-1, 1:-1] = True
        status, payload, info = _run_worker_with_timeout(
            _bench_worker.local_l2_2d_worker,
            (phi_win, phi_anchor_win, interior_mask, THRESHOLD, max_minimize_iter),
            timeout_s=local_timeout_s)
        if status == 'ok':
            phi_new = payload[0]
            phi[:, vy, vx] = phi_new
            stats['local_successes'] += 1
        elif status == 'timeout':
            stats['local_timeouts'] += 1
        else:
            stats['local_errors'] += 1
        # Stall detection on global fold count.
        T1, T2 = _triangle_areas_2d(phi[0], phi[1])
        n_neg_now = int((T1 <= 0).sum() + (T2 <= 0).sum())
        if last_n_neg is None or n_neg_now < last_n_neg:
            stall = 0
        else:
            stall += 1
        last_n_neg = n_neg_now
        stats['outer_iters_run'] = outer + 1
        if stall >= stall_limit:
            stats['stalled'] = True
            stats['n_neg_after'] = n_neg_now
            return phi, stats
    stats['n_neg_after'] = int(((T1 <= 0).sum() + (T2 <= 0).sum()))
    return phi, stats


def run_l2_phase(deformation_3channel):
    """Windowed L2 SLSQP with perturb-on-stall fallback.

    Returns ``(phi_l2, info)`` where info captures totals across all rounds.
    """
    t_start = time.time()
    phi = np.stack([deformation_3channel[1, 0].copy(),
                     deformation_3channel[2, 0].copy()])
    phi_anchor = phi.copy()
    H_, W_ = phi.shape[1:]
    info = {
        'wall_time': 0.0,
        'exception': None, 'timed_out': False,
        'outer_iters_total': 0,
        'local_timeouts_total': 0,
        'local_errors_total': 0,
        'local_successes_total': 0,
        'perturb_rounds': 0,
        'final_stalled': False,
    }
    try:
        for r in range(L2_MAX_PERTURB_ROUNDS + 1):
            phi, stats = _windowed_l2_pass(
                phi, phi_anchor,
                max_outer=L2_MAX_OUTER,
                max_minimize_iter=L2_MAX_MINIMIZE_ITER,
                local_timeout_s=L2_LOCAL_TIMEOUT_S,
                stall_limit=L2_STALL_LIMIT,
                log_prefix=f'r{r}:')
            info['outer_iters_total'] += stats['outer_iters_run']
            info['local_timeouts_total'] += stats['local_timeouts']
            info['local_errors_total'] += stats['local_errors']
            info['local_successes_total'] += stats['local_successes']
            T1, T2 = _triangle_areas_2d(phi[0], phi[1])
            if int((T1 <= 0).sum() + (T2 <= 0).sum()) == 0:
                break
            if not stats['stalled'] or r == L2_MAX_PERTURB_ROUNDS:
                break
            # Perturb-on-stall: add Gaussian noise to the cells that are
            # *currently folded* (their neighbourhood) so SLSQP gets a
            # fresh starting point. Same idea as notebook 14's reactive
            # warm-restart, applied at the slice level.
            cell_min = np.minimum(T1, T2)
            cell_fold_mask = cell_min <= THRESHOLD - ERR_TOL
            # Dilate the fold mask by 1 pixel so the perturbation covers
            # the corners participating in each folded triangle.
            from scipy.ndimage import binary_dilation
            perturb_mask = binary_dilation(cell_fold_mask, iterations=2)
            # Cell mask is (H-1, W-1); pad to (H, W) covering the corners.
            pixel_mask = np.zeros((H_, W_), dtype=bool)
            pixel_mask[:-1, :-1] |= perturb_mask
            pixel_mask[1:, :-1] |= perturb_mask
            pixel_mask[:-1, 1:] |= perturb_mask
            pixel_mask[1:, 1:] |= perturb_mask
            # Keep the boundary fixed (no perturbation on the field edges).
            pixel_mask[0, :] = False; pixel_mask[-1, :] = False
            pixel_mask[:, 0] = False; pixel_mask[:, -1] = False
            rng = np.random.default_rng(12345 + r)
            noise = rng.normal(scale=L2_PERTURB_SIGMA, size=(2, int(pixel_mask.sum())))
            phi[0][pixel_mask] += noise[0]
            phi[1][pixel_mask] += noise[1]
            info['perturb_rounds'] += 1
    except Exception as exc:  # noqa: BLE001
        info['exception'] = f'{type(exc).__name__}: {exc}'
    T1, T2 = _triangle_areas_2d(phi[0], phi[1])
    info['final_stalled'] = bool(int((T1 <= 0).sum() + (T2 <= 0).sum()) > 0)
    info['wall_time'] = time.time() - t_start
    return phi, info

In [ ]:
def run_l1_polish_phase(phi_l2, phi_anchor):
    """Windowed L1 polish over connected components of touched cells.

    Each component's local SLSQP runs inside a child process with a
    wall-clock timeout so an SLSQP hang on a pathological component can
    only cost ``L1_POLISH_TIMEOUT_S`` seconds, not the whole slice.

    Returns ``(phi_l1, info)``.
    """
    t0 = time.time()
    H_, W_ = phi_l2.shape[1:]
    diff_mag = np.abs(phi_l2 - phi_anchor).max(axis=0)
    touched = diff_mag > L1_TOUCH_TOL
    info = {
        'n_components': 0, 'n_components_polished': 0,
        'n_components_rejected': 0, 'n_components_skipped_too_large': 0,
        'n_components_timed_out': 0,
        'iter_total': 0, 'wall_time': 0.0,
    }
    if not touched.any():
        info['wall_time'] = time.time() - t0
        return phi_l2.copy(), info

    labels, n_comp = cc_label(touched)
    info['n_components'] = int(n_comp)
    phi_l1 = phi_l2.copy()

    for cid in range(1, n_comp + 1):
        cells = np.argwhere(labels == cid)
        y0, x0 = cells.min(axis=0)
        y1, x1 = cells.max(axis=0) + 1
        y0 = max(0, y0 - L1_BORDER); x0 = max(0, x0 - L1_BORDER)
        y1 = min(H_, y1 + L1_BORDER); x1 = min(W_, x1 + L1_BORDER)
        win_h = y1 - y0; win_w = x1 - x0
        if win_h * win_w > L1_COMPONENT_MAX_SIZE or win_h < 3 or win_w < 3:
            info['n_components_skipped_too_large'] += 1
            continue
        interior_mask = np.zeros((win_h, win_w), dtype=bool)
        interior_mask[1:-1, 1:-1] = True
        phi_win_l2 = phi_l1[:, y0:y1, x0:x1].copy()
        phi_win_anchor = phi_anchor[:, y0:y1, x0:x1].copy()
        status, payload, wi = _run_worker_with_timeout(
            _bench_worker.local_l1_2d_worker,
            (phi_win_l2, phi_win_anchor, interior_mask,
             THRESHOLD, EPS_L1, L1_POLISH_MAX_ITER),
            timeout_s=L1_POLISH_TIMEOUT_S,
        )
        if status != 'ok':
            if status == 'timeout':
                info['n_components_timed_out'] += 1
            info['n_components_rejected'] += 1
            continue
        phi_win_new, win_info = payload
        # Verify the candidate keeps the slice feasible AND drops L1.
        phi_l1_candidate = phi_l1.copy()
        phi_l1_candidate[:, y0:y1, x0:x1] = phi_win_new
        T1c, T2c = _triangle_areas_2d(phi_l1_candidate[0], phi_l1_candidate[1])
        if int((T1c <= 0).sum() + (T2c <= 0).sum()) > 0:
            info['n_components_rejected'] += 1
            continue
        l1_before = float(np.abs(phi_l1[:, y0:y1, x0:x1] - phi_win_anchor).sum())
        l1_after = float(np.abs(phi_win_new - phi_win_anchor).sum())
        if l1_after >= l1_before - 1e-9:
            info['n_components_rejected'] += 1
            continue
        phi_l1 = phi_l1_candidate
        info['n_components_polished'] += 1
        info['iter_total'] += win_info['nit']

    info['wall_time'] = time.time() - t0
    return phi_l1, info

## Per-slice runner

In [ ]:
def run_one_slice(phi_full, z):
    """Run the full L2 + L1 pipeline on slice z. Returns a result-row dict."""
    deformation = phi_full[:, z:z+1, :, :].copy()  # (3, 1, H, W)
    phi_anchor = np.stack([deformation[1, 0].copy(), deformation[2, 0].copy()])
    init = metrics_2d(phi_anchor, phi_anchor)
    t_total = time.time()
    row = {
        'z': int(z), 'H': int(H), 'W': int(W),
        'init_n_neg_tri': init['n_neg_tri'],
        'init_min_tri': init['min_tri'],
    }
    if init['n_neg_tri'] == 0:
        row.update({
            'L2_n_neg_tri': 0, 'L2_min_tri': init['min_tri'],
            'L2_L1': 0.0, 'L2_L2': 0.0, 'L2_t': 0.0,
            'L2_outer_iters': 0, 'L2_local_timeouts': 0,
            'L2_local_errors': 0, 'L2_perturb_rounds': 0,
            'L2_final_stalled': False, 'L2_exception': None,
            'L1_L1': 0.0, 'L1_L2': 0.0,
            'L1_n_neg_tri': 0, 'L1_min_tri': init['min_tri'],
            'L1_components': 0, 'L1_polished': 0,
            'L1_rejected': 0, 'L1_skipped_large': 0, 'L1_timed_out': 0,
            'L1_iter_total': 0, 'L1_t': 0.0,
            'total_t': time.time() - t_total,
            'feasible': True,
            'l1_drop_pct': 0.0,
            'phi_l1': np.stack([deformation[1, 0], deformation[2, 0]]),
        })
        return row
    phi_l2, l2_info = run_l2_phase(deformation)
    l2_m = metrics_2d(phi_l2, phi_anchor)
    row.update({
        'L2_n_neg_tri': l2_m['n_neg_tri'], 'L2_min_tri': l2_m['min_tri'],
        'L2_L1': l2_m['L1'], 'L2_L2': l2_m['L2'],
        'L2_t': l2_info['wall_time'],
        'L2_outer_iters': l2_info['outer_iters_total'],
        'L2_local_timeouts': l2_info['local_timeouts_total'],
        'L2_local_errors': l2_info['local_errors_total'],
        'L2_perturb_rounds': l2_info['perturb_rounds'],
        'L2_final_stalled': bool(l2_info['final_stalled']),
        'L2_exception': l2_info['exception'],
    })
    if l2_m['n_neg_tri'] == 0:
        phi_l1, l1_info = run_l1_polish_phase(phi_l2, phi_anchor)
        l1_m = metrics_2d(phi_l1, phi_anchor)
    else:
        phi_l1 = phi_l2
        l1_m = l2_m
        l1_info = {'n_components': 0, 'n_components_polished': 0,
                   'n_components_rejected': 0,
                   'n_components_skipped_too_large': 0,
                   'n_components_timed_out': 0,
                   'iter_total': 0, 'wall_time': 0.0}
    row.update({
        'L1_L1': l1_m['L1'], 'L1_L2': l1_m['L2'],
        'L1_n_neg_tri': l1_m['n_neg_tri'], 'L1_min_tri': l1_m['min_tri'],
        'L1_components': l1_info['n_components'],
        'L1_polished': l1_info['n_components_polished'],
        'L1_rejected': l1_info['n_components_rejected'],
        'L1_skipped_large': l1_info['n_components_skipped_too_large'],
        'L1_timed_out': l1_info['n_components_timed_out'],
        'L1_iter_total': l1_info['iter_total'],
        'L1_t': l1_info['wall_time'],
        'total_t': time.time() - t_total,
        'feasible': bool(l1_m['n_neg_tri'] == 0),
        'l1_drop_pct': (100.0 * (l2_m['L1'] - l1_m['L1']) / l2_m['L1']
                         if l2_m['L1'] > 0 else 0.0),
        'phi_l1': phi_l1,
    })
    return row

## Resumable orchestration

Reads `per_slice.csv` if it exists and skips any `z` already present. Appends one row per slice as it finishes. Snapshots the corrected volume to `checkpoint.npz` every `CHECKPOINT_EVERY` slices.

Each slice is wrapped in `try/except` so a single bad slice cannot crash the whole run -- failures are logged with `feasible=False` and the original field is kept for that slice.

In [ ]:
CSV_COLUMNS = [
    'z', 'H', 'W',
    'init_n_neg_tri', 'init_min_tri',
    'L2_n_neg_tri', 'L2_min_tri', 'L2_L1', 'L2_L2', 'L2_t',
    'L2_outer_iters', 'L2_local_timeouts', 'L2_local_errors',
    'L2_perturb_rounds', 'L2_final_stalled', 'L2_exception',
    'L1_L1', 'L1_L2', 'L1_n_neg_tri', 'L1_min_tri',
    'L1_components', 'L1_polished', 'L1_rejected',
    'L1_skipped_large', 'L1_timed_out',
    'L1_iter_total', 'L1_t',
    'total_t', 'feasible', 'l1_drop_pct',
]


def load_done_slices():
    if not os.path.exists(CSV_PATH):
        return set()
    try:
        df = pd.read_csv(CSV_PATH)
        return set(int(z) for z in df['z'].values)
    except Exception:
        return set()


def init_csv_if_needed():
    if not os.path.exists(CSV_PATH):
        with open(CSV_PATH, 'w', newline='') as f:
            f.write(','.join(CSV_COLUMNS) + '\n')


def append_csv_row(row):
    parts = []
    for c in CSV_COLUMNS:
        v = row.get(c, '')
        if v is None:
            parts.append('')
        elif isinstance(v, float):
            parts.append(f'{v:.6g}')
        elif isinstance(v, bool):
            parts.append('True' if v else 'False')
        else:
            s = str(v).replace(',', ';').replace('\n', ' ')
            parts.append(s)
    with open(CSV_PATH, 'a', newline='') as f:
        f.write(','.join(parts) + '\n')


def save_checkpoint(corrected_full):
    tmp = CKPT_PATH + '.tmp'
    np.savez_compressed(tmp, phi_corrected=corrected_full)
    os.replace(tmp, CKPT_PATH)


def load_checkpoint(default):
    if os.path.exists(CKPT_PATH):
        with np.load(CKPT_PATH) as data:
            arr = data['phi_corrected']
            if arr.shape == default.shape:
                return arr.copy()
    return default.copy()


def log_line(msg):
    print(msg, flush=True)
    with open(LOG_PATH, 'a') as f:
        f.write(msg + '\n')


init_csv_if_needed()
done = load_done_slices()
corrected_full = load_checkpoint(phi_full)
log_line(f'[start] {time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}  '
         f'already-done = {len(done)} / {D}')

In [ ]:
if SMOKE_TEST:
    if SMOKE_SLICE_INDICES is not None:
        target_slices = list(SMOKE_SLICE_INDICES)
    else:
        folded = np.where(fold_per_slice > 0)[0]
        if len(folded) == 0:
            target_slices = [0]
        else:
            order = np.argsort(fold_per_slice[folded])
            picks = [folded[order[0]], folded[order[len(order)//2]], folded[order[-1]]]
            target_slices = [int(z) for z in picks]
    log_line(f'[smoke] running slices {target_slices}')
else:
    target_slices = list(range(D))


def _placeholder_row(z, init_n_neg, init_min, exc_str, total_t):
    return {
        'z': int(z), 'H': int(H), 'W': int(W),
        'init_n_neg_tri': init_n_neg, 'init_min_tri': init_min,
        'L2_n_neg_tri': -1, 'L2_min_tri': float('nan'),
        'L2_L1': float('nan'), 'L2_L2': float('nan'),
        'L2_t': 0.0,
        'L2_outer_iters': 0, 'L2_local_timeouts': 0, 'L2_local_errors': 0,
        'L2_perturb_rounds': 0, 'L2_final_stalled': True,
        'L2_exception': exc_str,
        'L1_L1': float('nan'), 'L1_L2': float('nan'),
        'L1_n_neg_tri': -1, 'L1_min_tri': float('nan'),
        'L1_components': 0, 'L1_polished': 0,
        'L1_rejected': 0, 'L1_skipped_large': 0, 'L1_timed_out': 0,
        'L1_iter_total': 0, 'L1_t': 0.0,
        'total_t': total_t, 'feasible': False,
        'l1_drop_pct': 0.0, 'phi_l1': None,
    }


for i, z in enumerate(target_slices):
    if z in done:
        continue
    t_slice = time.time()
    try:
        row = run_one_slice(corrected_full, z)
    except Exception as exc:
        tb = traceback.format_exc(limit=4).replace('\n', ' | ')
        log_line(f'[ERR  ] z={z}  {type(exc).__name__}: {exc} :: {tb}')
        n_neg, mt = slice_fold_stats(corrected_full, z)
        row = _placeholder_row(z, n_neg, mt,
                                f'{type(exc).__name__}: {exc}',
                                time.time() - t_slice)
    if row.get('phi_l1') is not None:
        corrected_full[1, z] = row['phi_l1'][0]
        corrected_full[2, z] = row['phi_l1'][1]
    row_to_write = {k: v for k, v in row.items() if k != 'phi_l1'}
    append_csv_row(row_to_write)
    done.add(z)
    log_line(f'[slice] z={z:>4d}  init={row["init_n_neg_tri"]:>5d}  '
              f'L2_neg={row["L2_n_neg_tri"]:>4d}  '
              f'L1_neg={row["L1_n_neg_tri"]:>4d}  '
              f'L2_L1={row["L2_L1"]:>9.3f}  L1_L1={row["L1_L1"]:>9.3f}  '
              f'drop={row["l1_drop_pct"]:>5.1f}%  '
              f'L2_outer={row["L2_outer_iters"]:>4d} pert={row["L2_perturb_rounds"]}  '
              f'L2_t={row["L2_t"]:>6.1f}s L1_t={row["L1_t"]:>5.1f}s  '
              f'feas={row["feasible"]}')
    if (i + 1) % CHECKPOINT_EVERY == 0:
        save_checkpoint(corrected_full)
        log_line(f'[ckpt ] saved at i={i+1}, done={len(done)}/{D}')

save_checkpoint(corrected_full)
log_line(f'[end  ] {time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}  done={len(done)}/{D}')

## Aggregate stats

In [ ]:
df = pd.read_csv(CSV_PATH)
df_folded = df[df['init_n_neg_tri'] > 0].copy()
df_feasible = df_folded[df_folded['feasible']].copy()

print(f'total slices                            : {len(df)}')
print(f'slices with folds (work needed)         : {len(df_folded)}')
print(f'slices reaching feasibility             : {len(df_feasible)} / {len(df_folded)}'
      + (f'  ({100.0*len(df_feasible)/len(df_folded):.1f}%)' if len(df_folded) else ''))
print()
if len(df_feasible) > 0:
    print(f'L1 drop after polish (feasible slices only):')
    print(f'  mean = {df_feasible["l1_drop_pct"].mean():.2f}%   '
          f'median = {df_feasible["l1_drop_pct"].median():.2f}%   '
          f'max = {df_feasible["l1_drop_pct"].max():.2f}%')
    print(f'L2 phase wall time (feasible slices):')
    print(f'  total = {df_feasible["L2_t"].sum():.1f}s   '
          f'mean = {df_feasible["L2_t"].mean():.1f}s   '
          f'p95 = {df_feasible["L2_t"].quantile(0.95):.1f}s   '
          f'max = {df_feasible["L2_t"].max():.1f}s')
    print(f'L1 polish wall time (feasible slices):')
    print(f'  total = {df_feasible["L1_t"].sum():.1f}s   '
          f'mean = {df_feasible["L1_t"].mean():.1f}s   '
          f'p95 = {df_feasible["L1_t"].quantile(0.95):.1f}s   '
          f'max = {df_feasible["L1_t"].max():.1f}s')
    print(f'Wall time grand total: {df["total_t"].sum():.1f}s'
          f' = {df["total_t"].sum() / 3600:.2f} h')

In [ ]:
FIG_DIR = OUTPUT_DIR

# (1) per-slice fold overview
fig, ax = plt.subplots(figsize=(11, 3.5), constrained_layout=True)
ax.plot(df['z'], df['init_n_neg_tri'], lw=0.7, color='#c62828', label='initial n_neg_tri')
ax.plot(df['z'], df['L1_n_neg_tri'].clip(lower=0), lw=0.7, color='#1b8a3a',
         label='after L2+L1 (n_neg_tri)')
ax.set_yscale('symlog', linthresh=1)
ax.set_xlabel('z (slice index)'); ax.set_ylabel('n_neg_tri (symlog)')
ax.set_title('Folded triangles per slice: before vs after L2+L1')
ax.legend()
fig.savefig(os.path.join(FIG_DIR, 'per_slice_fold_overview.png'), dpi=200, bbox_inches='tight')
fig.savefig(os.path.join(FIG_DIR, 'per_slice_fold_overview.pdf'), bbox_inches='tight')
plt.show()

# (2) L1 vs L2 sparsity scatter
if len(df_feasible) > 0:
    fig, ax = plt.subplots(figsize=(6.5, 6), constrained_layout=True)
    lo = min(df_feasible['L1_L1'].min(), df_feasible['L2_L1'].min()) * 0.95
    hi = max(df_feasible['L1_L1'].max(), df_feasible['L2_L1'].max()) * 1.05
    ax.plot([lo, hi], [lo, hi], color='#888888', lw=1, linestyle='--')
    ax.scatter(df_feasible['L2_L1'], df_feasible['L1_L1'],
                s=8, alpha=0.7, color='#1b8a3a')
    ax.set_xlabel('L1-norm of correction after L2 phase'); ax.set_ylabel('L1-norm after L2+L1 phase')
    ax.set_title('L1 norm: L2 result vs L1-polished result (one point per feasible slice)\n'
                  'Points below the diagonal = L1 polish strictly improves the correction')
    ax.set_aspect('equal')
    fig.savefig(os.path.join(FIG_DIR, 'l1_vs_l2_sparsity.png'), dpi=200, bbox_inches='tight')
    fig.savefig(os.path.join(FIG_DIR, 'l1_vs_l2_sparsity.pdf'), bbox_inches='tight')
    plt.show()

# (3) time distribution
if len(df_folded) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), constrained_layout=True)
    axes[0].hist(df_folded['L2_t'], bins=60, color='#5b7fb5')
    axes[0].set_xlabel('L2 phase time (s)'); axes[0].set_ylabel('# slices')
    axes[0].set_title('L2 phase runtime per folded slice')
    axes[1].hist(df_folded['L1_t'], bins=60, color='#1b8a3a')
    axes[1].set_xlabel('L1 polish time (s)'); axes[1].set_ylabel('# slices')
    axes[1].set_title('L1 polish runtime per folded slice')
    fig.savefig(os.path.join(FIG_DIR, 'time_distribution.png'), dpi=200, bbox_inches='tight')
    fig.savefig(os.path.join(FIG_DIR, 'time_distribution.pdf'), bbox_inches='tight')
    plt.show()

# (4) worst-slice visual
if len(df_folded) > 0:
    worst_z = int(df_folded.sort_values('init_n_neg_tri', ascending=False).iloc[0]['z'])
    phi_init_slice = np.stack([phi_full[1, worst_z].copy(),
                                 phi_full[2, worst_z].copy()])
    phi_l1_slice = np.stack([corrected_full[1, worst_z].copy(),
                               corrected_full[2, worst_z].copy()])
    res = np.linalg.norm(phi_l1_slice - phi_init_slice, axis=0)
    fig, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
    T1i, T2i = _triangle_areas_2d(phi_init_slice[0], phi_init_slice[1])
    T1f, T2f = _triangle_areas_2d(phi_l1_slice[0], phi_l1_slice[1])
    init_min = np.minimum(T1i, T2i)
    final_min = np.minimum(T1f, T2f)
    vmax = max(abs(init_min.min()), abs(final_min.min()), 1.0)
    axes[0].imshow(init_min, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    axes[0].set_title(f'worst slice z={worst_z}: min(T1,T2) before')
    axes[0].set_xticks([]); axes[0].set_yticks([])
    axes[1].imshow(final_min, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    axes[1].set_title('after L2+L1 polish')
    axes[1].set_xticks([]); axes[1].set_yticks([])
    im2 = axes[2].imshow(res, cmap='Reds')
    axes[2].set_title('per-pixel correction magnitude')
    axes[2].set_xticks([]); axes[2].set_yticks([])
    fig.colorbar(im2, ax=axes[2], shrink=0.85)
    fig.savefig(os.path.join(FIG_DIR, 'worst_slice_visual.png'), dpi=200, bbox_inches='tight')
    fig.savefig(os.path.join(FIG_DIR, 'worst_slice_visual.pdf'), bbox_inches='tight')
    plt.show()

## Save final corrected volume

In [ ]:
final_path = os.path.join(OUTPUT_DIR, 'deformation3d_corrected_2d.npy')
np.save(final_path, corrected_full)
print(f'Saved corrected (per-slice 2D) volume to {final_path}')

summary = {
    'data_path': DATA_PATH,
    'shape': list(phi_full.shape),
    'threshold': THRESHOLD, 'eps_l1': EPS_L1,
    'L2_max_outer': L2_MAX_OUTER,
    'L2_max_per_index': L2_MAX_PER_INDEX,
    'L2_max_minimize': L2_MAX_MINIMIZE,
    'L1_polish_max_iter': L1_POLISH_MAX_ITER,
    'L1_touch_tol': L1_TOUCH_TOL,
    'L1_border': L1_BORDER,
    'L1_component_max_size': L1_COMPONENT_MAX_SIZE,
    'total_slices': int(len(df)),
    'slices_with_folds': int(len(df_folded)),
    'slices_feasible': int(len(df_feasible)),
    'l1_drop_pct_mean': float(df_feasible['l1_drop_pct'].mean()) if len(df_feasible) else 0.0,
    'l1_drop_pct_median': float(df_feasible['l1_drop_pct'].median()) if len(df_feasible) else 0.0,
    'total_wall_time_s': float(df['total_t'].sum()),
}
with open(os.path.join(OUTPUT_DIR, 'summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)
print('Saved summary.json')